# Experimento Formal de Clustering para Representaciones Textuales

## Objetivo general
Comparar 4 variantes de clustering implementadas en `validation/clustering.py` sobre 3 representaciones textuales (BoW, TF-IDF y embeddings Nomic) y 3 datasets (DialogSum, StackOverflow, ESQAD).

## Preguntas de investigación
1. ¿Qué combinación algoritmo-distancia logra mejor equilibrio entre cohesión y separación (ASW, CH)?
2. ¿Cuándo las métricas difusas (PC, PE, XB) aportan evidencia adicional respecto al clustering duro?
3. ¿Cómo cambia el rendimiento al pasar de representaciones dispersas (BoW/TF-IDF) a embeddings densos (Nomic)?

## Hipótesis iniciales
1. Las variantes con distancia coseno serán competitivas en datos textuales.
2. FCM capturará mejor ambiguedad temática en datasets heterogéneos.
3. Los embeddings Nomic tenderán a mayor separabilidad semántica, con mayor coste computacional.

## Diseño metodológico

Diseño factorial completo:
- 4 algoritmos: KMeans-euclidean, KMeans-cosine, FCM-euclidean, FCM-cosine
- 3 representaciones: BoW, TF-IDF, Nomic embeddings
- 3 datasets: DialogSum, StackOverflow, ESQAD
- N semillas por condición

Total de corridas = 4 x 3 x 3 x |K| x |semillas|.

Se incluye modo piloto para validar pipeline extremo a extremo antes de lanzar el barrido completo.

In [1]:
# Imports y configuración global
from __future__ import annotations

import importlib.util
import json
import logging
import os
import pickle
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

# Asegura que el root del repo este en sys.path, incluso si el notebook se ejecuta desde validation/notebooks
CANDIDATE_ROOT = Path.cwd().resolve()
if CANDIDATE_ROOT.name == "notebooks" and CANDIDATE_ROOT.parent.name == "validation":
    ROOT = CANDIDATE_ROOT.parent.parent
else:
    ROOT = next(
        (p for p in [CANDIDATE_ROOT, *CANDIDATE_ROOT.parents] if (p / "validation" / "clustering.py").exists()),
        CANDIDATE_ROOT,
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def _load_local_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

# Carga explícita de módulos locales para evitar colisiones con paquetes externos llamados "validation"
clustering_module = _load_local_module("tfg_validation_clustering", ROOT / "validation" / "clustering.py")
datasets_module = _load_local_module("tfg_validation_datasets", ROOT / "validation" / "datasets.py")
metrics_module = _load_local_module("tfg_validation_metrics", ROOT / "validation" / "metrics" / "metrics.py")
emb_module = _load_local_module("tfg_validation_ollama_embeddings", ROOT / "validation" / "representation" / "ollama_embeddings.py")

GenericKMeans = clustering_module.GenericKMeans
DatasetLoader = datasets_module.DatasetLoader
evaluate_fuzzy_clustering = metrics_module.evaluate_fuzzy_clustering
evaluate_hard_clustering = metrics_module.evaluate_hard_clustering
OllamaEmbeddings = emb_module.OllamaEmbeddings

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print(f"ROOT detectado: {ROOT}")
print(f"UMAP disponible: {HAS_UMAP}")

/home/gabrielsanchez/repos/TFG-Chatbot/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT detectado: /home/gabrielsanchez/repos/TFG-Chatbot
UMAP disponible: True


In [2]:
# Configuración de logging y rutas de salida
RESULTS_DIR = ROOT / "validation" / "results" / "clustering_experiment"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts"
LOGS_DIR = RESULTS_DIR / "logs"

for p in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR, ARTIFACTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

log_file = LOGS_DIR / "experiment.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("clustering_experiment")

def save_table(df: pd.DataFrame, filename: str) -> Path:
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    return path

def save_figure(fig: plt.Figure, filename: str, dpi: int = 200) -> Path:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path

print(f"Resultados en: {RESULTS_DIR}")

Resultados en: /home/gabrielsanchez/repos/TFG-Chatbot/validation/results/clustering_experiment


In [3]:
# Carga de datasets
loader = DatasetLoader()

# Ajusta límites si deseas acelerar en fase piloto
PILOT_MODE = True
PILOT_LIMITS = {
    "dialogsum": 1200,
    "stackoverflow": 1200,
    "esquad": 1200,
}

df_dialogsum = loader.load_dialogsum()
df_stack = loader.load_stackoverflow(limit=PILOT_LIMITS["stackoverflow"] if PILOT_MODE else 5000)
df_esquad = loader.load_esquad()

if PILOT_MODE:
    df_dialogsum = df_dialogsum.head(PILOT_LIMITS["dialogsum"])
    df_esquad = df_esquad.head(PILOT_LIMITS["esquad"])

for df, name in [
    (df_dialogsum, "dialogsum"),
    (df_stack, "stackoverflow"),
    (df_esquad, "esquad"),
]:
    df["dataset"] = name

datasets_raw = {
    "dialogsum": df_dialogsum[["text", "label", "dataset"]].copy(),
    "stackoverflow": df_stack[["text", "label", "dataset"]].copy(),
    "esquad": df_esquad[["text", "label", "dataset"]].copy(),
}

{k: v.shape for k, v in datasets_raw.items()}

Loading DialogSum...
Loading Stack Overflow subset...
Loading ESQAD (Spanish)...


{'dialogsum': (1200, 3), 'stackoverflow': (1200, 3), 'esquad': (1190, 3)}

In [4]:
# Limpieza y control de calidad
MIN_TEXT_LEN = 15

def clean_dataset(df: pd.DataFrame, min_len: int = 15) -> pd.DataFrame:
    out = df.copy()
    out = out.dropna(subset=["text"])
    out["text"] = out["text"].astype(str).str.strip()
    out = out[out["text"].str.len() > 0]
    out = out.drop_duplicates(subset=["text"])
    out = out[out["text"].str.len() >= min_len]
    return out.reset_index(drop=True)

datasets = {name: clean_dataset(df, min_len=MIN_TEXT_LEN) for name, df in datasets_raw.items()}
{k: v.shape for k, v in datasets.items()}

{'dialogsum': (1200, 3), 'stackoverflow': (1125, 3), 'esquad': (1184, 3)}

## Reporte descriptivo de muestra

En la siguiente celda se reportan, por dataset:
1. Número de documentos
2. Longitud media del texto
3. Número de etiquetas reales observadas

In [5]:
report_rows = []
for name, df in datasets.items():
    report_rows.append({
        "dataset": name,
        "n_docs": len(df),
        "avg_text_len": float(df["text"].str.len().mean()),
        "n_unique_labels": int(df["label"].nunique()),
    })

df_sample_report = pd.DataFrame(report_rows).sort_values("dataset")
save_table(df_sample_report, "sample_report.csv")
df_sample_report

,dataset,n_docs,avg_text_len,n_unique_labels
0,dialogsum,1200,740.635833,1006
2,esquad,1184,68.238176,1
1,stackoverflow,1125,199.427556,1


In [6]:
# Funciones de representación
def build_bow(texts: list[str], params: dict) -> np.ndarray:
    start = time.perf_counter()
    vectorizer = CountVectorizer(**params)
    X = vectorizer.fit_transform(texts)
    X = X.astype(np.float64)
    elapsed = time.perf_counter() - start
    return X, vectorizer, elapsed

def build_tfidf(texts: list[str], params: dict) -> np.ndarray:
    start = time.perf_counter()
    vectorizer = TfidfVectorizer(**params)
    X = vectorizer.fit_transform(texts)
    X = X.astype(np.float64)
    elapsed = time.perf_counter() - start
    return X, vectorizer, elapsed

def build_nomic_embeddings(texts: list[str], batch_size: int, model_name: str, host: str = "localhost", port: int = 11434):
    start = time.perf_counter()
    embedder = OllamaEmbeddings(model=model_name, host=host, port=port, timeout=60.0)
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        Xb = embedder.embed_batch(batch, normalize=True)
        vectors.append(Xb)
    X = np.vstack(vectors) if vectors else np.zeros((len(texts), 768), dtype=np.float32)
    elapsed = time.perf_counter() - start
    return X, embedder, elapsed

In [7]:
# Configuración por representación
REP_CONFIG = {
    "bow": {
        "max_features": 3000 if not PILOT_MODE else 1200,
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
    },
    "tfidf": {
        "max_features": 3000 if not PILOT_MODE else 1200,
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True,
    },
    "nomic": {
        "model_name": "nomic-embed-text",
        "batch_size": 64 if PILOT_MODE else 128,
        "host": os.getenv("OLLAMA_HOST", "localhost"),
        "port": int(os.getenv("OLLAMA_PORT", "11434")),
    },
}
REP_CONFIG

{'bow': {'max_features': 1200,
  'ngram_range': (1, 2),
  'min_df': 2,
  'max_df': 0.95},
 'tfidf': {'max_features': 1200,
  'ngram_range': (1, 2),
  'min_df': 2,
  'max_df': 0.95,
  'sublinear_tf': True},
 'nomic': {'model_name': 'nomic-embed-text',
  'batch_size': 64,
  'host': '172.19.80.1',
  'port': 11434}}

In [8]:
# Construcción efectiva y cache de representaciones
representations: dict[str, dict[str, np.ndarray]] = {}
representation_meta: list[dict] = []

for ds_name, df in datasets.items():
    texts = df["text"].tolist()
    representations[ds_name] = {}

    # BoW
    bow_cache = ARTIFACTS_DIR / f"{ds_name}_bow.pkl"
    if bow_cache.exists():
        with open(bow_cache, "rb") as f:
            payload = pickle.load(f)
        X_bow = payload["X"]
        bow_elapsed = payload.get("elapsed_sec", np.nan)
    else:
        X_bow, bow_vec, bow_elapsed = build_bow(texts, REP_CONFIG["bow"])
        X_bow = normalize(X_bow, norm="l2", axis=1)
        with open(bow_cache, "wb") as f:
            pickle.dump({"X": X_bow, "vectorizer": bow_vec, "elapsed_sec": bow_elapsed}, f)
    representations[ds_name]["bow"] = X_bow
    representation_meta.append({"dataset": ds_name, "representation": "bow", "shape": str(X_bow.shape), "build_sec": bow_elapsed})

    # TF-IDF
    tfidf_cache = ARTIFACTS_DIR / f"{ds_name}_tfidf.pkl"
    if tfidf_cache.exists():
        with open(tfidf_cache, "rb") as f:
            payload = pickle.load(f)
        X_tfidf = payload["X"]
        tfidf_elapsed = payload.get("elapsed_sec", np.nan)
    else:
        X_tfidf, tfidf_vec, tfidf_elapsed = build_tfidf(texts, REP_CONFIG["tfidf"])
        X_tfidf = normalize(X_tfidf, norm="l2", axis=1)
        with open(tfidf_cache, "wb") as f:
            pickle.dump({"X": X_tfidf, "vectorizer": tfidf_vec, "elapsed_sec": tfidf_elapsed}, f)
    representations[ds_name]["tfidf"] = X_tfidf
    representation_meta.append({"dataset": ds_name, "representation": "tfidf", "shape": str(X_tfidf.shape), "build_sec": tfidf_elapsed})

    # Nomic
    nomic_cache = ARTIFACTS_DIR / f"{ds_name}_nomic.npy"
    nomic_meta = ARTIFACTS_DIR / f"{ds_name}_nomic_meta.json"
    if nomic_cache.exists() and nomic_meta.exists():
        X_nomic = np.load(nomic_cache)
        meta = json.loads(nomic_meta.read_text())
        nomic_elapsed = meta.get("elapsed_sec", np.nan)
    else:
        try:
            X_nomic, embedder, nomic_elapsed = build_nomic_embeddings(
                texts=texts,
                batch_size=REP_CONFIG["nomic"]["batch_size"],
                model_name=REP_CONFIG["nomic"]["model_name"],
                host=REP_CONFIG["nomic"]["host"],
                port=REP_CONFIG["nomic"]["port"],
            )
            X_nomic = normalize(X_nomic, norm="l2", axis=1)
            np.save(nomic_cache, X_nomic)
            nomic_meta.write_text(json.dumps({"elapsed_sec": nomic_elapsed}))
        except Exception as e:
            logger.warning(f"No se pudieron generar embeddings Nomic para {ds_name}: {e}")
            X_nomic = None
            nomic_elapsed = np.nan
    representations[ds_name]["nomic"] = X_nomic
    representation_meta.append({"dataset": ds_name, "representation": "nomic", "shape": str(None if X_nomic is None else X_nomic.shape), "build_sec": nomic_elapsed})

df_rep_meta = pd.DataFrame(representation_meta)
save_table(df_rep_meta, "representation_build_report.csv")
df_rep_meta

,dataset,representation,shape,build_sec
0,dialogsum,bow,"(1200, 1200)",0.168985
1,dialogsum,tfidf,"(1200, 1200)",0.172550
2,dialogsum,nomic,"(1200, 768)",29.843220
3,stackoverflow,bow,"(1125, 1200)",0.033507
4,stackoverflow,tfidf,"(1125, 1200)",0.031609
5,stackoverflow,nomic,"(1125, 768)",31.365638
6,esquad,bow,"(1184, 1200)",0.018171
7,esquad,tfidf,"(1184, 1200)",0.016087
8,esquad,nomic,"(1184, 768)",29.827756


## Política de hiperparámetros

Parámetros base:
- $k \in [2, 15]$ (acotado por tamaño del dataset)
- semillas: 3 en piloto, 5 en ejecución completa
- `max_iter = 200`, `tol = 1e-4`
- FCM con `m = 2.0`

## Criterio de comparación exploratoria

Criterio principal:
- `ASW` (mayor es mejor)

Criterios de apoyo:
- `CH` (mayor es mejor)
- `runtime_sec` (menor es mejor)
- `n_iter` (menor es mejor)
- Para FCM: `XB` (menor), `PC` (mayor), `PE` (menor)

Regla de desempate para selección final:
1. Mayor `asw_mean`
2. Menor `runtime_sec_mean`
3. Menor `n_iter_mean`

In [9]:
# Grid experimental
K_VALUES = list(range(2, 8)) if PILOT_MODE else list(range(2, 16))
SEEDS = [42, 52, 62] if PILOT_MODE else [42, 52, 62, 72, 82]

ALGO_CONFIGS = [
    {"algorithm": "kmeans", "distance": "euclidean", "m": None},
    {"algorithm": "kmeans", "distance": "cosine", "m": None},
    {"algorithm": "fcm", "distance": "euclidean", "m": 2.0},
    {"algorithm": "fcm", "distance": "cosine", "m": 2.0},
]

rows = []
for ds_name, rep_dict in representations.items():
    for rep_name, X in rep_dict.items():
        if X is None:
            continue
        n = X.shape[0]
        k_values_ds = [k for k in K_VALUES if k < n]
        for cfg in ALGO_CONFIGS:
            for k in k_values_ds:
                for seed in SEEDS:
                    rows.append({
                        "dataset": ds_name,
                        "representation": rep_name,
                        "algorithm": cfg["algorithm"],
                        "distance": cfg["distance"],
                        "m": cfg["m"],
                        "k": k,
                        "seed": seed,
                    })

df_grid = pd.DataFrame(rows)
save_table(df_grid, "experiment_grid.csv")
df_grid.head(), len(df_grid)

(     dataset representation algorithm   distance   m  k  seed
 0  dialogsum            bow    kmeans  euclidean NaN  2    42
 1  dialogsum            bow    kmeans  euclidean NaN  2    52
 2  dialogsum            bow    kmeans  euclidean NaN  2    62
 3  dialogsum            bow    kmeans  euclidean NaN  3    42
 4  dialogsum            bow    kmeans  euclidean NaN  3    52,
 648)

In [10]:
# Runner principal
def _to_dense_if_needed(X):
    return X.toarray() if hasattr(X, "toarray") else X

def run_single_experiment(
    X,
    dataset_name: str,
    representation_name: str,
    algorithm: str,
    distance: str,
    k: int,
    seed: int,
    m: float | None = None,
    max_iter: int = 200,
    tol: float = 1e-4,
) -> dict:
    X_arr = _to_dense_if_needed(X)

    model = GenericKMeans(
        n_clusters=k,
        algorithm=algorithm,
        distance=distance,
        max_iter=max_iter,
        tol=tol,
        random_state=seed,
        m=2.0 if m is None else m,
    )

    start = time.perf_counter()
    model.fit(X_arr)
    elapsed = time.perf_counter() - start

    if algorithm == "fcm":
        metrics = evaluate_fuzzy_clustering(
            X_arr,
            model.membership_,
            model.centroids_,
            m=2.0 if m is None else m,
            distance=distance,
        )
    else:
        metrics = evaluate_hard_clustering(X_arr, model.labels_, metric=distance)

    return {
        "dataset": dataset_name,
        "representation": representation_name,
        "algorithm": algorithm,
        "distance": distance,
        "m": m,
        "k": k,
        "seed": seed,
        "asw": metrics.asw,
        "ch": metrics.ch,
        "pc": metrics.pc,
        "pe": metrics.pe,
        "xb": metrics.xb,
        "inertia": model.inertia_,
        "n_iter": model.n_iter_,
        "runtime_sec": elapsed,
    }

In [11]:
# Bucle completo del experimento
results = []
errors = []

for _, row in df_grid.iterrows():
    ds = row["dataset"]
    rep = row["representation"]
    X = representations[ds][rep]

    try:
        out = run_single_experiment(
            X=X,
            dataset_name=ds,
            representation_name=rep,
            algorithm=row["algorithm"],
            distance=row["distance"],
            k=int(row["k"]),
            seed=int(row["seed"]),
            m=row["m"],
        )
        results.append(out)
    except Exception as e:
        errors.append({
            "dataset": ds,
            "representation": rep,
            "algorithm": row["algorithm"],
            "distance": row["distance"],
            "k": int(row["k"]),
            "seed": int(row["seed"]),
            "error": str(e),
        })

df_results = pd.DataFrame(results)
df_errors = pd.DataFrame(errors)

if not df_results.empty:
    save_table(df_results, "raw_results.csv")
if not df_errors.empty:
    save_table(df_errors, "execution_errors.csv")

df_results.head(), len(df_results), len(df_errors)

2026-04-15 09:36:03,434 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=euclidean
2026-04-15 09:36:03,631 | INFO | tfg_validation_clustering | K-Means converged at iteration 14
2026-04-15 09:36:03,632 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=euclidean, inertia=814.4248
2026-04-15 09:36:27,866 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=euclidean
2026-04-15 09:36:28,126 | INFO | tfg_validation_clustering | K-Means converged at iteration 12
2026-04-15 09:36:28,130 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=euclidean, inertia=814.4183
2026-04-15 09:36:38,560 | INFO | tfg_validation_clustering | Fitting K-Means with k=2, distance=euclidean
2026-04-15 09:36:38,795 | INFO | tfg_validation_clustering | K-Means converged at iteration 17
2026-04-15 09:36:38,808 | INFO | tfg_validation_clustering | Clustering finished: algorithm=kmeans, distance=euclidean, ine

(     dataset representation algorithm   distance   m  k  seed       asw  \
 0  dialogsum            bow    kmeans  euclidean NaN  2    42  0.025764   
 1  dialogsum            bow    kmeans  euclidean NaN  2    52  0.025601   
 2  dialogsum            bow    kmeans  euclidean NaN  2    62  0.025516   
 3  dialogsum            bow    kmeans  euclidean NaN  3    42  0.021109   
 4  dialogsum            bow    kmeans  euclidean NaN  3    52  0.022236   
 
           ch  pc  pe  xb     inertia  n_iter  runtime_sec  
 0  39.707823 NaN NaN NaN  814.424805      14     0.199887  
 1  39.717680 NaN NaN NaN  814.418318      12     0.287113  
 2  39.719913 NaN NaN NaN  814.416849      17     0.250347  
 3  30.535877 NaN NaN NaN  800.573202      17     0.396181  
 4  30.582112 NaN NaN NaN  800.514364      17     0.326126  ,
 648,
 0)

In [12]:
# Agregación por condición (media, desviación estándar, IC95)
def ci95(series: pd.Series) -> float:
    s = series.dropna()
    if len(s) <= 1:
        return np.nan
    return 1.96 * float(s.std(ddof=1)) / np.sqrt(len(s))

group_cols = ["dataset", "representation", "algorithm", "distance", "k"]
metric_cols = ["asw", "ch", "pc", "pe", "xb", "runtime_sec", "n_iter"]

agg_dict = {}
for mcol in metric_cols:
    agg_dict[f"{mcol}_mean"] = (mcol, "mean")
    agg_dict[f"{mcol}_std"] = (mcol, "std")
    agg_dict[f"{mcol}_ci95"] = (mcol, ci95)

df_summary = df_results.groupby(group_cols).agg(**agg_dict).reset_index()
save_table(df_summary, "summary_by_condition.csv")
df_summary.head()

,dataset,representation,algorithm,distance,k,asw_mean,asw_std,asw_ci95,ch_mean,ch_std,...,pe_ci95,xb_mean,xb_std,xb_ci95,runtime_sec_mean,runtime_sec_std,runtime_sec_ci95,n_iter_mean,n_iter_std,n_iter_ci95
0,dialogsum,bow,fcm,cosine,2,0.042328,0.006688,0.007568,28.278747,5.729707,...,0.000008,4.248336e+09,3.713930e+09,4.202707e+09,0.051624,0.021354,0.024164,2.0,0.0,0.0
1,dialogsum,bow,fcm,cosine,3,0.018942,0.016117,0.018238,19.517717,1.597411,...,0.000021,6.175847e+11,9.120574e+11,1.032090e+12,0.055258,0.042155,0.047703,2.0,0.0,0.0
2,dialogsum,bow,fcm,cosine,4,0.008031,0.008026,0.009082,17.995585,0.749943,...,0.000012,2.545936e+10,1.449989e+10,1.640817e+10,0.100062,0.039133,0.044284,2.0,0.0,0.0
3,dialogsum,bow,fcm,cosine,5,0.008276,0.010689,0.012096,14.048526,1.374915,...,0.000011,1.724459e+10,1.991616e+10,2.253726e+10,0.183487,0.107271,0.121389,2.0,0.0,0.0
4,dialogsum,bow,fcm,cosine,6,0.014454,0.008905,0.010077,14.419720,0.632490,...,0.000002,5.111695e+10,3.492976e+10,3.952675e+10,0.133284,0.033680,0.038112,2.0,0.0,0.0


In [13]:
# Ranking y selección
# Criterio principal: ASW alto, con desempate por runtime y n_iter
df_rank = df_summary.copy()

df_rank["rank_asw"] = df_rank.groupby(["dataset", "representation"])["asw_mean"].rank(ascending=False, method="min")
df_rank["rank_runtime"] = df_rank.groupby(["dataset", "representation"])["runtime_sec_mean"].rank(ascending=True, method="min")
df_rank["rank_n_iter"] = df_rank.groupby(["dataset", "representation"])["n_iter_mean"].rank(ascending=True, method="min")

df_rank = df_rank.sort_values([
    "dataset", "representation", "asw_mean", "runtime_sec_mean", "n_iter_mean"
], ascending=[True, True, False, True, True])

df_best_per_condition = (
    df_rank
    .groupby(["dataset", "representation"], as_index=False, sort=False)
    .first()
)

df_best_global = df_rank.sort_values(["asw_mean", "runtime_sec_mean", "n_iter_mean"], ascending=[False, True, True]).head(20)

save_table(df_rank, "summary_ranked_full.csv")
save_table(df_best_per_condition, "summary_best_per_condition.csv")
save_table(df_best_global, "summary_best_global_top20.csv")

df_best_per_condition

,dataset,representation,algorithm,distance,k,asw_mean,asw_std,asw_ci95,ch_mean,ch_std,...,xb_ci95,runtime_sec_mean,runtime_sec_std,runtime_sec_ci95,n_iter_mean,n_iter_std,n_iter_ci95,rank_asw,rank_runtime,rank_n_iter
0,dialogsum,bow,kmeans,cosine,2,0.054909,0.002353,0.002662,37.108066,0.921304,...,4.202707e+09,0.218593,0.056025,0.063398,10.000000,1.732051,1.960000,1.0,8.0,13.0
1,dialogsum,nomic,kmeans,cosine,6,0.058320,0.002259,0.002556,25.004423,0.851163,...,5.534891e+10,0.146684,0.043571,0.049305,12.000000,4.000000,4.526426,1.0,18.0,17.0
2,dialogsum,tfidf,kmeans,cosine,2,0.019155,0.002574,0.002913,11.215528,2.233373,...,7.178826e+11,0.191217,0.025486,0.028840,8.000000,1.000000,1.131607,1.0,10.0,13.0
3,esquad,bow,kmeans,cosine,3,0.072946,0.007117,0.008053,39.927445,1.701253,...,4.031977e+06,0.306722,0.084627,0.095765,20.333333,6.350853,7.186667,1.0,18.0,19.0
4,esquad,nomic,kmeans,cosine,2,0.112006,0.025034,0.028329,37.110051,0.497625,...,1.772451e+11,0.118733,0.020969,0.023728,9.666667,2.081666,2.355627,1.0,13.0,14.0
5,esquad,tfidf,kmeans,cosine,7,0.034507,0.001211,0.001371,9.456995,0.297738,...,3.498419e+11,0.389276,0.146164,0.165400,20.000000,9.643651,10.912818,1.0,18.0,18.0
6,stackoverflow,bow,kmeans,cosine,2,0.072169,0.000294,0.000332,50.451955,0.496143,...,8.393458e+06,0.212056,0.014190,0.016058,15.333333,1.527525,1.728558,1.0,15.0,19.0
7,stackoverflow,nomic,kmeans,cosine,7,0.045596,0.003352,0.003793,16.519057,0.943606,...,7.636003e+10,0.191311,0.056097,0.063479,18.333333,6.806859,7.702686,1.0,18.0,18.0
8,stackoverflow,tfidf,kmeans,cosine,7,0.018942,0.001788,0.002023,6.285730,0.329934,...,5.169336e+11,0.302153,0.086252,0.097604,17.000000,4.000000,4.526426,1.0,19.0,20.0


In [14]:
# Resumen exploratorio global (sin inferencia estadística)
df_summary_explore = df_summary.copy()
df_summary_explore["variant"] = df_summary_explore["algorithm"] + "_" + df_summary_explore["distance"]

df_global_by_variant = (
    df_summary_explore
    .groupby(["algorithm", "distance", "representation"], as_index=False)
    .agg(
        asw_mean=("asw_mean", "mean"),
        ch_mean=("ch_mean", "mean"),
        runtime_sec_mean=("runtime_sec_mean", "mean"),
        n_iter_mean=("n_iter_mean", "mean"),
        xb_mean=("xb_mean", "mean"),
        pc_mean=("pc_mean", "mean"),
        pe_mean=("pe_mean", "mean"),
    )
)

df_top_configs_by_asw = (
    df_summary_explore
    .sort_values(["asw_mean", "runtime_sec_mean", "n_iter_mean"], ascending=[False, True, True])
    .head(20)
    .reset_index(drop=True)
)

save_table(df_best_per_condition, "summary_best_per_condition.csv")
save_table(df_global_by_variant, "summary_global_by_variant.csv")
save_table(df_top_configs_by_asw, "summary_top_configs_by_asw.csv")

display(df_best_per_condition.head(10))
display(df_global_by_variant.sort_values(["asw_mean", "runtime_sec_mean"], ascending=[False, True]).head(10))
df_top_configs_by_asw.head(10)

,dataset,representation,algorithm,distance,k,asw_mean,asw_std,asw_ci95,ch_mean,ch_std,...,xb_ci95,runtime_sec_mean,runtime_sec_std,runtime_sec_ci95,n_iter_mean,n_iter_std,n_iter_ci95,rank_asw,rank_runtime,rank_n_iter
0,dialogsum,bow,kmeans,cosine,2,0.054909,0.002353,0.002662,37.108066,0.921304,...,4.202707e+09,0.218593,0.056025,0.063398,10.000000,1.732051,1.960000,1.0,8.0,13.0
1,dialogsum,nomic,kmeans,cosine,6,0.058320,0.002259,0.002556,25.004423,0.851163,...,5.534891e+10,0.146684,0.043571,0.049305,12.000000,4.000000,4.526426,1.0,18.0,17.0
2,dialogsum,tfidf,kmeans,cosine,2,0.019155,0.002574,0.002913,11.215528,2.233373,...,7.178826e+11,0.191217,0.025486,0.028840,8.000000,1.000000,1.131607,1.0,10.0,13.0
3,esquad,bow,kmeans,cosine,3,0.072946,0.007117,0.008053,39.927445,1.701253,...,4.031977e+06,0.306722,0.084627,0.095765,20.333333,6.350853,7.186667,1.0,18.0,19.0
4,esquad,nomic,kmeans,cosine,2,0.112006,0.025034,0.028329,37.110051,0.497625,...,1.772451e+11,0.118733,0.020969,0.023728,9.666667,2.081666,2.355627,1.0,13.0,14.0
5,esquad,tfidf,kmeans,cosine,7,0.034507,0.001211,0.001371,9.456995,0.297738,...,3.498419e+11,0.389276,0.146164,0.165400,20.000000,9.643651,10.912818,1.0,18.0,18.0
6,stackoverflow,bow,kmeans,cosine,2,0.072169,0.000294,0.000332,50.451955,0.496143,...,8.393458e+06,0.212056,0.014190,0.016058,15.333333,1.527525,1.728558,1.0,15.0,19.0
7,stackoverflow,nomic,kmeans,cosine,7,0.045596,0.003352,0.003793,16.519057,0.943606,...,7.636003e+10,0.191311,0.056097,0.063479,18.333333,6.806859,7.702686,1.0,18.0,18.0
8,stackoverflow,tfidf,kmeans,cosine,7,0.018942,0.001788,0.002023,6.285730,0.329934,...,5.169336e+11,0.302153,0.086252,0.097604,17.000000,4.000000,4.526426,1.0,19.0,20.0


,algorithm,distance,representation,asw_mean,ch_mean,runtime_sec_mean,n_iter_mean,xb_mean,pc_mean,pe_mean
6,kmeans,cosine,bow,0.061152,29.466829,0.314059,15.277778,NaN,NaN,NaN
7,kmeans,cosine,nomic,0.049291,24.240951,0.138631,11.796296,NaN,NaN,NaN
9,kmeans,euclidean,bow,0.030737,31.534195,0.664037,21.277778,NaN,NaN,NaN
10,kmeans,euclidean,nomic,0.025693,24.915030,0.453126,24.259259,NaN,NaN,NaN
0,fcm,cosine,bow,0.025575,21.143020,0.060373,2.000000,4.271032e+10,0.265507,1.420800
1,fcm,cosine,nomic,0.021467,17.855675,0.029237,2.000000,7.875226e+11,0.265479,1.420855
8,kmeans,cosine,tfidf,0.018757,8.344156,0.341628,16.092593,NaN,NaN,NaN
2,fcm,cosine,tfidf,0.010375,6.277891,0.049395,2.481481,2.650357e+12,0.265477,1.420859
11,kmeans,euclidean,tfidf,0.009678,8.851254,0.525421,18.222222,NaN,NaN,NaN
4,fcm,euclidean,nomic,0.007291,18.774779,0.079507,3.870370,9.450562e+10,0.265476,1.420860


,dataset,representation,algorithm,distance,k,asw_mean,asw_std,asw_ci95,ch_mean,ch_std,...,xb_mean,xb_std,xb_ci95,runtime_sec_mean,runtime_sec_std,runtime_sec_ci95,n_iter_mean,n_iter_std,n_iter_ci95,variant
0,esquad,nomic,kmeans,cosine,2,0.112006,0.025034,0.028329,37.110051,0.497625,...,NaN,NaN,NaN,0.118733,0.020969,0.023728,9.666667,2.081666,2.355627,kmeans_cosine
1,esquad,bow,kmeans,cosine,3,0.072946,0.007117,0.008053,39.927445,1.701253,...,NaN,NaN,NaN,0.306722,0.084627,0.095765,20.333333,6.350853,7.186667,kmeans_cosine
2,stackoverflow,bow,kmeans,cosine,2,0.072169,0.000294,0.000332,50.451955,0.496143,...,NaN,NaN,NaN,0.212056,0.014190,0.016058,15.333333,1.527525,1.728558,kmeans_cosine
3,stackoverflow,bow,kmeans,cosine,7,0.071461,0.004375,0.004950,19.112242,0.371054,...,NaN,NaN,NaN,0.250177,0.049772,0.056322,14.666667,3.214550,3.637606,kmeans_cosine
4,stackoverflow,bow,kmeans,cosine,4,0.070393,0.000569,0.000644,27.991855,2.195555,...,NaN,NaN,NaN,0.260070,0.075368,0.085287,16.000000,5.291503,5.987899,kmeans_cosine
5,stackoverflow,bow,kmeans,cosine,3,0.069964,0.002771,0.003136,33.393721,1.941883,...,NaN,NaN,NaN,0.161566,0.036594,0.041410,11.333333,3.055050,3.457115,kmeans_cosine
6,esquad,bow,kmeans,cosine,5,0.069618,0.005029,0.005690,31.267535,0.436142,...,NaN,NaN,NaN,0.265257,0.067816,0.076741,15.000000,4.358899,4.932558,kmeans_cosine
7,esquad,bow,kmeans,cosine,4,0.069243,0.006044,0.006839,35.677327,0.463766,...,NaN,NaN,NaN,0.218378,0.034778,0.039355,12.666667,2.081666,2.355627,kmeans_cosine
8,esquad,bow,kmeans,cosine,6,0.069088,0.004701,0.005320,29.523937,1.186934,...,NaN,NaN,NaN,0.354927,0.073246,0.082885,17.000000,3.605551,4.080065,kmeans_cosine
9,stackoverflow,bow,kmeans,cosine,6,0.069009,0.004431,0.005014,20.603959,1.204358,...,NaN,NaN,NaN,0.276284,0.117584,0.133058,13.666667,3.785939,4.284193,kmeans_cosine
